# 直接使用 Python 运行 ADLab

本教程展示如何不通过 YAML 配置文件，直接使用 Python 代码运行 ADLab。

## 目录
1. [准备工作](#1-准备工作)
2. [创建 Config](#2-创建-config)
3. [创建 LLM Provider](#3-创建-llm-provider)
4. [创建 Evaluator](#4-创建-evaluator)
5. [运行搜索](#5-运行搜索)
6. [完整示例](#6-完整示例)

## 1. 准备工作

In [ ]:
# 设置 API Key
import os
os.environ["OPENAI_API_KEY"] = "your-api-key"  # 替换为你的 API Key

### 模板程序

In [ ]:
# 模板程序是算法的骨架，包含函数签名和部分实现
template_code = '''
from typing import List

def sort_list(lst: List[int]) -> List[int]:
    """
    Sort a list of integers in ascending order.

    Args:
        lst: A list of integers to sort

    Returns:
        A new list with integers sorted in ascending order
    """
    # TODO: Implement the sorting algorithm
    pass
'''
print("模板程序:")
print(template_code)

### 任务描述

In [ ]:
# 任务描述告诉 LLM 要解决什么问题
task_description = '''
You are an expert programmer. Your task is to implement a sorting algorithm.

Requirements:
1. The function should take a list of integers
2. Return a new list sorted in ascending order
3. Do not modify the original list
4. Handle edge cases: empty list, single element

Example:
Input: [3, 1, 4, 1, 5]
Output: [1, 1, 3, 4, 5]

Important:
- Do not use built-in sorted() or list.sort()
- Implement your own sorting algorithm (e.g., bubble sort, quick sort)
'''
print("任务描述:")
print(task_description)

## 2. 创建 Config

In [ ]:
from algodisco.methods.funsearch.config import FunSearchConfig

# 创建 FunSearchConfig 对象
config = FunSearchConfig(
    template_program=template_code,  # 模板程序代码（字符串）
    task_description=task_description,  # 任务描述（字符串）
    language="python",
    num_samplers=2,  # 并行采样数
    num_evaluators=2,  # 并行评估数
    examples_per_prompt=2,  # 提示中的示例数
    samples_per_prompt=2,  # 每次提示生成的样本数
    max_samples=100,  # 最大样本数
    llm_max_tokens=1024,  # LLM 生成的最大 token 数
    llm_timeout_seconds=60,  # LLM 超时时间（秒）
    
    # 数据库配置
    db_num_islands=5,  # 岛屿数量
    db_reset_period=7200,  # 重置周期（秒）
)

print("Config 创建成功!")
print(f"template_program 长度: {len(config.template_program)} 字符")
print(f"max_samples: {config.max_samples}")

## 3. 创建 LLM Provider

In [ ]:
from algodisco.providers.llm.openai_api import OpenAIAPI

# 方式1: 使用 OpenAI
llm = OpenAIAPI(
    model="gpt-3.5-turbo",
    api_key=os.environ["OPENAI_API_KEY"],
    # base_url="https://api.openai.com/v1",  # 默认值
)

# 方式2: 使用其他 LLM Provider（如 Claude、vLLM、SGLang）
# from algodisco.providers.llm.claude_api import ClaudeAPI
# llm = ClaudeAPI(
#     model="claude-3-sonnet-20240229",
#     api_key=os.environ["ANTHROPIC_API_KEY"],
# )

# from algodisco.providers.llm.vllm import vLLM
# llm = vLLM(
#     model="llama-3-8b",
#     api_base="http://localhost:8000/v1",
# )

print("LLM Provider 创建成功!")
print(f"Model: {llm.model}")

## 4. 创建 Evaluator

In [ ]:
# 创建自定义 Evaluator
import random
import subprocess
import tempfile
from algodisco.base.evaluator import Evaluator, EvalResult

class SortingEvaluator(Evaluator):
    """评估排序算法的正确性"""
    
    def __init__(self, num_tests=100):
        self.num_tests = num_tests
        self.test_cases = self._generate_test_cases()
    
    def _generate_test_cases(self):
        """生成随机测试用例"""
        cases = []
        for _ in range(self.num_tests):
            n = random.randint(0, 20)
            case = random.sample(range(-100, 100), n)
            expected = sorted(case)
            cases.append((case, expected))
        return cases
    
    def evaluate_program(self, program_str: str) -> EvalResult:
        """评估程序"""
        try:
            # 写入临时文件
            with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
                f.write(program_str)
                f.flush()
                temp_path = f.name
            
            # 测试
            correct = 0
            for inputs, expected in self.test_cases[:20]:
                test_code = f"""
import sys
sys.path.insert(0, '.')
exec(open(\"{temp_path}\").read())
result = sort_list({inputs})
print(result == {expected})
"""
                exec_result = subprocess.run(
                    ['python', '-c', test_code],
                    capture_output=True,
                    timeout=5
                )
                if exec_result.returncode == 0 and 'True' in exec_result.stdout.decode():
                    correct += 1
            
            score = correct / min(20, len(self.test_cases))
            
            return {
                "score": score,
                "execution_time": 0.0,
                "error_msg": None
            }
            
        except subprocess.TimeoutExpired:
            return {
                "score": 0.0,
                "execution_time": 5.0,
                "error_msg": "Timeout"
            }
        except Exception as e:
            return {
                "score": 0.0,
                "execution_time": 0.0,
                "error_msg": str(e)[:200]
            }

# 创建 Evaluator 实例
evaluator = SortingEvaluator(num_tests=100)
print("Evaluator 创建成功!")
print(f"测试用例数: {evaluator.num_tests}")

## 5. 运行搜索

In [ ]:
from algodisco.methods.funsearch.search import FunSearch

# 创建 FunSearch 实例
search = FunSearch(
    config=config,
    llm=llm,
    evaluator=evaluator,
    # logger=logger,  # 可选：添加日志记录器
)

print("FunSearch 实例创建成功!")
print("调用 search.run() 开始搜索...")

In [ ]:
# 运行搜索（取消注释以执行）
# search.run()

# 获取搜索结果
# best_program = search._database.get_best_program()
# print(f"最佳得分: {best_program.score}")
# print(f"最佳程序:\n{best_program.program}")

# 获取所有评估过的程序
# all_programs = search._database.get_all_programs()
# for p in sorted(all_programs, key=lambda x: x.score, reverse=True)[:5]:
#     print(f"得分: {p.score}, 程序: {p.program[:100]}...")

## 6. 完整示例

In [ ]:
"""
完整示例：将以上所有代码整合在一起
只需提供 API Key 即可运行
"""

import os
import random
import subprocess
import tempfile

# 设置 API Key
os.environ["OPENAI_API_KEY"] = "your-api-key"  # 替换为你的 API Key

# 导入模块
from algodisco.methods.funsearch.config import FunSearchConfig
from algodisco.methods.funsearch.search import FunSearch
from algodisco.providers.llm.openai_api import OpenAIAPI
from algodisco.base.evaluator import Evaluator

# ===== 1. 准备数据 =====
template_code = '''
from typing import List

def sort_list(lst: List[int]) -> List[int]:
    """Sort a list of integers in ascending order."""
    pass
'''

task_description = '''
Implement a sorting algorithm. Do not use built-in sorted() or list.sort().
'''

# ===== 2. 创建 Config =====
config = FunSearchConfig(
    template_program=template_code,
    task_description=task_description,
    language="python",
    num_samplers=2,
    num_evaluators=2,
    samples_per_prompt=2,
    max_samples=10,  # 测试时使用较小的值
    llm_max_tokens=512,
)

# ===== 3. 创建 LLM =====
llm = OpenAIAPI(
    model="gpt-3.5-turbo",
    api_key=os.environ["OPENAI_API_KEY"],
)

# ===== 4. 创建 Evaluator =====
class SimpleEvaluator(Evaluator):
    def __init__(self):
        self.test_cases = [
            ([3, 1, 2], [1, 2, 3]),
            ([5, 4, 3, 2, 1], [1, 2, 3, 4, 5]),
            ([], []),
            ([1], [1]),
        ]
    
    def evaluate_program(self, program_str):
        try:
            with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
                f.write(program_str)
                f.flush()
                temp_path = f.name
            
            correct = 0
            for inputs, expected in self.test_cases:
                test_code = f"""
exec(open(\"{temp_path}\").read())
result = sort_list({inputs})
print(result == {expected})
"""
                result = subprocess.run(['python', '-c', test_code], capture_output=True, timeout=5)
                if result.returncode == 0 and 'True' in result.stdout.decode():
                    correct += 1
            
            return {"score": correct / len(self.test_cases), "execution_time": 0.0, "error_msg": None}
        except Exception as e:
            return {"score": 0.0, "execution_time": 0.0, "error_msg": str(e)[:200]}

evaluator = SimpleEvaluator()

# ===== 5. 运行搜索 =====
# search = FunSearch(config=config, llm=llm, evaluator=evaluator)
# search.run()

print("完整示例代码已准备好！")
print("取消注释最后两行代码即可运行搜索")

## 总结

直接使用 Python 运行 ADLab 的步骤：

1. **准备 template_program 和 task_description** - 作为字符串传入
2. **创建 FunSearchConfig** - 配置搜索参数
3. **创建 LLM Provider** - 选择 OpenAI、Claude、vLLM 等
4. **创建 Evaluator** - 继承 Evaluator 基类，实现 evaluate_program 方法
5. **创建 FunSearch 实例** - 传入 config、llm、evaluator
6. **调用 search.run()** - 开始搜索